In [1]:
!pip install torch transformers peft accelerate datasets evaluate pandas


In [10]:
"""
BERT + PEFT LoRA 微调正反例二分类
数据集：csv文件，两列：text(文本), label(1=正例，0=负例)
运行前：accelerate config （单机GPU/CPU直接回车默认配置即可）
"""
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from evaluate import load

In [11]:
MODEL_NAME = "bert-base-chinese"  # 中文用这个；英文换成 bert-base-uncased
MAX_SEQ_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-4
OUTPUT_DIR = "./bert-lora-rerank"

# LoRA 配置，BERT分类任务
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],  # BERT attention中的Q/V层
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"  # 序列二分类
)

In [12]:
# ========== 2. 加载你的正反例数据集 ==========
# pos_neg_dataset.csv：text, label
df = pd.read_csv("pos_neg_dataset.csv")
raw_ds = Dataset.from_pandas(df)

# 划分训练集 90% / 验证集10%
split_ds = raw_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
val_ds = split_ds["test"]

In [13]:
# ==========3. Tokenizer 文本编码 ==========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_SEQ_LEN
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [14]:
# ==========4. 加载BERT基础模型 + LoRA ==========
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,  # 二分类：正例/反例
    torch_dtype=torch.float32
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() # 打印可训练参数占比，LoRA一般<1%

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 296,450 || all params: 102,565,636 || trainable%: 0.2890


In [15]:
# ==========5. 评估指标 ==========
metric = load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [17]:
# ==========6. 训练参数 ==========
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,  # 保留验证集最优模型
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(), # GPU自动开启半精度；CPU自动关闭
    report_to="none",
    save_total_limit=2
)

In [18]:
# ==========7. Trainer 启动训练 ==========
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

if __name__ == "__main__":
    # 启动训练，accelerate自动接管设备（CPU/GPU）
    trainer.train()
    # 保存LoRA适配器（体积很小，几十MB）
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"LoRA权重保存路径：{OUTPUT_DIR}")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.697266,0.500000
2,No log,0.698242,0.500000
3,No log,0.697754,0.500000


[transformers] mtime may not be reliable on this filesystem, falling back to numerical ordering


LoRA权重保存路径：./bert-lora-rerank
